In [ ]:
# Libraries

## data
import pandas as pd
pd.set_option('display.max_columns', None)
import json
from uuid import uuid4
import re

## utility
from datetime import datetime
from tqdm import tqdm

## LLMs
from ollama import chat, create, AsyncClient
from pydantic import BaseModel

## Async
import asyncio
import nest_asyncio
nest_asyncio.apply()




### Data


In [ ]:
# Load data
df_test = pd.read_excel("../development_dataset/dev_jobapplication.xlsx")
df_test.head(3)

,Job,Question,Format,Item,Response
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,NaN
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,NaN
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,NaN


In [ ]:
# Load job ads dataset
df_ads = pd.read_excel("../development_dataset/dev_companies_jobs_and_ads.xlsx")
df_merged = pd.merge(left=df_test, right=df_ads, left_on='Job', right_on='Role Title', how='left')
df_merged['id'] = df_merged['Advertisement'].apply(lambda x: uuid4().__str__().split('-')[-1][:5])
df_merged.head(3)

,Job,Question,Format,Item,Response,Company,Mission,Products,Annual Revenue in Billions,Employee Count,Role Title,Department,Department Function,Advertisement,id
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,NaN,SteelForge,Shaping the future with innovative metal solut...,"Steel products, alloys, metal fabrication serv...",$30,"50,000",Maintenance Planner,Maintenance & Reliability,Equipment Maintenance,**Company Overview:** \nSteelForge is a leade...,60ab4
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,NaN,SteelForge,Shaping the future with innovative metal solut...,"Steel products, alloys, metal fabrication serv...",$30,"50,000",Maintenance Planner,Maintenance & Reliability,Equipment Maintenance,**Company Overview:** \nSteelForge is a leade...,a9e62
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,NaN,SteelForge,Shaping the future with innovative metal solut...,"Steel products, alloys, metal fabrication serv...",$30,"50,000",Maintenance Planner,Maintenance & Reliability,Equipment Maintenance,**Company Overview:** \nSteelForge is a leade...,ce7cf


In [ ]:
df_merged[df_merged['Item'] == 'Resume']

,Job,Question,Format,Item,Response,Company,Mission,Products,Annual Revenue in Billions,Employee Count,Role Title,Department,Department Function,Advertisement,id
120,Maintenance Planner,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,SteelForge,Shaping the future with innovative metal solut...,"Steel products, alloys, metal fabrication serv...",$30,"50,000",Maintenance Planner,Maintenance & Reliability,Equipment Maintenance,**Company Overview:** \nSteelForge is a leade...,d41c6
379,Chief Learning Officer,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,EduTech,Transforming lives through learning,"Educational textbooks, online courses, learnin...",$5,"30,000",Chief Learning Officer,Educational Services,Student Support,**Company Overview:**\n\nEduTech is at the for...,f12f9
619,VP of Engineering,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,SteelForge,Shaping the future with innovative metal solut...,"Steel products, alloys, metal fabrication serv...",$30,"50,000",VP of Engineering,Process Engineering,Process Design,**Company Overview** \nSteelForge is at the f...,32c67
901,Regulatory Affairs Manager,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,EnergyX,Powering progress with responsible energy solu...,"Petroleum products, renewable energy systems, ...",$200,"100,000",Regulatory Affairs Manager,Regulatory & Legal,Legal Compliance,EnergyX is a pioneering company dedicated to p...,19a0f
1189,Sustainability Manager,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,EnergyX,Powering progress with responsible energy solu...,"Petroleum products, renewable energy systems, ...",$200,"100,000",Sustainability Manager,Environmental & Safety,Environmental Compliance,**Company Overview** \nEnergyX is at the fore...,69978
1475,Logistics Director,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,EnergyX,Powering progress with responsible energy solu...,"Petroleum products, renewable energy systems, ...",$200,"100,000",Logistics Director,Supply Chain & Logistics,Procurement,**Company Overview** \nEnergyX is at the fore...,13d58
1703,Medical Billing Specialist,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,BlueInsurance,Empowering healthier lives through accessible ...,"Health insurance plans, pharmacy services, med...",$280,"350,000",Medical Billing Specialist,Healthcare Administration,Patient Administration,**Company Overview** \nBlueInsurance is dedic...,41895
1982,Senior Train Operator,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,RailTech,Delivering efficient and sustainable transport...,"Freight trains, passenger trains, rail infrast...",$20,"40,000",Senior Train Operator,Rail Operations,Train Operations,RailTech is a pioneering company dedicated to ...,64f83
2201,Defense Systems Director,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,AeroMech,Advancing global infrastructure and transporta...,"Commercial aircraft, construction equipment, d...",$90,"150,000",Defense Systems Director,Management,Executive Leadership,AeroMech is a pioneering company dedicated to ...,103e1
2504,Compensation Analyst,Please provide the text of your resume. The Re...,Text (Max 3000 characters),Resume,NaN,PrimeBuy,Innovating for a connected world,"Search engine, e-commerce platform, cloud serv...",$350,"1,000,000",Compensation Analyst,Human Resources,Talent Acquisition,PrimeBuy is a leading innovator in the digital...,cc951


In [ ]:
df_aux = df_test.copy()
df_aux['_q_type'] = df_aux['Item'].apply(lambda x: x.split()[0].strip().lower())

In [ ]:
df_aux.groupby('_q_type')['Format'].unique()


_q_type
cognitive       [Text  (Max 20 characters), Integer]
interview                [Text (Max 750 characters)]
job                               [Integer (1 to 5)]
personality    [Integer (1 to 5), Integer (-2 to 2)]
resume                  [Text (Max 3000 characters)]
Name: Format, dtype: object

In [ ]:
df_aux['Format'].unique()


array(['Integer (1 to 5)', 'Text (Max 3000 characters)',
       'Integer (-2 to 2)', 'Text (Max 750 characters)',
       'Text  (Max 20 characters)', 'Integer'], dtype=object)

In [ ]:
df_merged['Job'].unique()

array(['Maintenance Planner', 'Chief Learning Officer',
       'VP of Engineering', 'Regulatory Affairs Manager',
       'Sustainability Manager', 'Logistics Director',
       'Medical Billing Specialist', 'Senior Train Operator',
       'Defense Systems Director', 'Compensation Analyst', 'Radiologist',
       'Quality Systems Analyst', 'Research Coordinator', 'Station Staff',
       'Chief Safety Officer', 'Senior Nutritionist', 'Route Planner',
       'Product Planner', 'Account Coordinator',
       'Autonomous Systems Lead', 'Legal Research Associate',
       'Chief Medical Information Officer',
       'Clinical Informatics Director', 'Senior Route Planner',
       'Chief Revenue Officer', 'Research Lead', 'Pharmacist',
       'Training Coordinator', 'Quality Inspector',
       'Senior Food Scientist', 'Director of Food Safety',
       'Passenger Assistant', 'Plant Manager', 'Senior Loan Officer',
       'Educational Researcher', 'Food Service Worker', 'Nutritionist',
       'Branch

In [ ]:
def categorize_jobs(job_titles):
    """
    Categorize job titles into three levels:
      - clerical: entry-level or lower than mid-level.
      - middle_management: middle-level job positions.
      - chief_excutive: high-level positions (e.g. those with Chief, VP, or Director in the title).

    Args:
        job_titles (list of str): List of job titles.

    Returns:
        dict: Dictionary with keys 'clerical', 'middle_management', 'chief_excutive'
              mapping to lists of corresponding job titles.
    """
    categories = {
        "clerical": [],
        "middle_management": [],
        "chief_executive": []
    }

    for title in job_titles:
        title_lower = title.lower()

        # Check for high-level keywords first.
        if "chief" in title_lower or "vp" in title_lower or "director" in title_lower:
            categories["chief_executive"].append(title)
        # Check for middle-level management roles.
        elif "manager" in title_lower or "lead" in title_lower:
            categories["middle_management"].append(title)
        else:
            # Everything else is assumed to be clerical.
            categories["clerical"].append(title)

    return categories


# Example usage:
job_titles = [
    'Maintenance Planner', 'Chief Learning Officer', 'VP of Engineering',
    'Regulatory Affairs Manager', 'Sustainability Manager', 'Logistics Director',
    'Medical Billing Specialist', 'Senior Train Operator', 'Defense Systems Director',
    'Compensation Analyst', 'Radiologist', 'Quality Systems Analyst', 'Research Coordinator',
    'Station Staff', 'Chief Safety Officer', 'Senior Nutritionist', 'Route Planner',
    'Product Planner', 'Account Coordinator', 'Autonomous Systems Lead',
    'Legal Research Associate', 'Chief Medical Information Officer',
    'Clinical Informatics Director', 'Senior Route Planner', 'Chief Revenue Officer',
    'Research Lead', 'Pharmacist', 'Training Coordinator', 'Quality Inspector',
    'Senior Food Scientist', 'Director of Food Safety', 'Passenger Assistant',
    'Plant Manager', 'Senior Loan Officer', 'Educational Researcher',
    'Food Service Worker', 'Nutritionist', 'Branch Teller', 'Plant Operator',
    'Train Operator', 'Engineering Director', 'Director of Regulatory Affairs',
    'Legal Assistant', 'Product Developer', 'Food Scientist', 'Registered Nurse',
    'Physician Assistant', 'VP of Product', 'Training Director', 'Loan Officer',
    'Ticket Agent', 'Customer Service Representative', 'Director of Data Science'
]

categorized_jobs = categorize_jobs(job_titles)
for level, jobs in categorized_jobs.items():
    print(f"{level}:")
    for job in jobs:
        print(f"  - {job}")


clerical:
  - Maintenance Planner
  - Medical Billing Specialist
  - Senior Train Operator
  - Compensation Analyst
  - Radiologist
  - Quality Systems Analyst
  - Research Coordinator
  - Station Staff
  - Senior Nutritionist
  - Route Planner
  - Product Planner
  - Account Coordinator
  - Legal Research Associate
  - Senior Route Planner
  - Pharmacist
  - Training Coordinator
  - Quality Inspector
  - Senior Food Scientist
  - Passenger Assistant
  - Senior Loan Officer
  - Educational Researcher
  - Food Service Worker
  - Nutritionist
  - Branch Teller
  - Plant Operator
  - Train Operator
  - Legal Assistant
  - Product Developer
  - Food Scientist
  - Registered Nurse
  - Physician Assistant
  - Loan Officer
  - Ticket Agent
  - Customer Service Representative
middle_management:
  - Regulatory Affairs Manager
  - Sustainability Manager
  - Autonomous Systems Lead
  - Research Lead
  - Plant Manager
chief_executive:
  - Chief Learning Officer
  - VP of Engineering
  - Logistics 

### MODEL


In [ ]:
persona_directory = {
    "clerical": """
Your name is Susan Wilshire, 25 years old who lives in the Seattle, WA metropolitan area.
Persona: You are a recent university graduate with several years of experience in your field. You are working to provide for your family of a spouse and two children. As such you are a very motivated, dedicated, and hard working individual. Currently you are looking for a new job and are in the process of participating in job interviews.
Problem Statement: Your job is to answer job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for.
""",

"middle_management": """
Your name is Tina Wilshire, 34 years old who lives in the Washington D.C. metropolitan area.
You are an experienced leader with strong communication, strategic thinking, and team management skills. You balance high-level directives with practical execution, ensuring team success. You have a bachelor’s or master’s degree and a proven track record of leading teams effectively. You’re actively seeking a new job to advance your career and provide stability for your family. You take pride in your ability to lead teams effectively, solve problems proactively, and contribute to the company’s success in its vision. """,

"chief_executive": """
Your name is Henry Wilshire, 40 years old who lives in the Washington D.C. metropolitan area.
You are a senior level executive with multiple years of experience in your field (i.e., 15+ years).
You hold a bachelor's degree in finance and a Masters of Business Administration (MBA) from a top 20 MBA program according to US News. Your MBA coursework had a strong focus in international business strategy, strategic leadership, leadership development, and business intelligence analytics.
You worked full-time as an equity research analyst for approximately 5 years in-between your Bachelors and MBA so as to gain industry experience before going back to school. Additionally, during this 5 year time period between your bachelors and MBA, you also completed all three levels of the Chartered Financial Analyst (CFA) exam.
You have successfully managed multiple international and cross-discipline teams across a number of different business units (e.g., business intelligence, accounting, etc) throughout your career (particularly since earning your MBA).
You are highly competent in your leadership abilities and have attended multiple leadership training throughout your career. You take pride in your ability to lead teams effectively, solve problems proactively, and contribute to the firm’s success in its vision.
Early in your executive experience, while leading your team, you initially underestimated the challenge but quickly adapted by restructuring your own and the firm's priorities.
You originally planned on entering consulting when done with your MBA but transitioned into strategic leadership because you enjoyed cross-functional problem-solving.
Regarding work-life balance: Early in your career, you struggled with work-life balance but have since learned how to lead firms effectively while maintaining personal priorities.
Currently, you are looking for a new executive level position and actively participating in interviews, seeking an opportunity to grow your career.
"""
}

In [ ]:
system_prompt = '''
To help you with the process of landing a job, you are participating in an educational job interview mock-up. Your job is to answer job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for.

To provide an answer, you MUST follow these instructions:

Step 1- Consider your persona to answer the questions.
Step 2- Analyze the content of the JSON object presented. Identify the question presented to you.
If the question_type is 'Personality', continue to step 3.
If the question_type is 'Cognitive', skip to step 4.
If the question_type is 'Job', skip to step 5.
If the question_type is 'Interview', skip to step 6
If the question_type is 'Resume', skip to step 7.
Step 3- Based on your persona, with several years of experience, indicate how helpful such a personality trait would be in that job position. Continue to step 7.
Step 4- First, analyze the question and determine if it is a letter riddle skip to Step 3a or number riddle skip to Step 3b.
Step 5a- For a letter riddle, unscramble the letters and determine what word or words can be made with the letters. Skip to step 7.
Step 5b- For a number riddle, analyze the pattern and provide the correct answer to the question. Skip to step 7.
Step 6- For job readiness questions, think about whether or not such familiarity would be helpful in that job position. Skip to step 7.
Step 7- For this question, pretend that you have already gotten the job and are now faced with a scenario provided by the question. Provide a written response within 750 characters on how you would respond to the situation with best practice. Check the length of your response. REVISE if needed to make it less than 750 characters.
Step 8- Your educational background is in the field related to the job position. Provide your educational background, your applicable major, and applicable job experiences that you had prior to interviewing for this position. Make sure that the information is clear and concise, and the length is within the limits indicated. Continue to step 7.
Step 9- Check the format of your answer to match with the format indicated in the provided information. For questions with the text format, make sure your response is not longer than the specified limit. For the questions with the integer format required for the response, make sure your response is an integer. If a range is given, make sure your response fits within the range. Revise your response if necessary. Skip to step 8.
Step 10- Ensure you answered the question fully. Check the length of your answer is NOT longer than it should be. Your response MUST include the answer and the provided id. Then repeat the process for the next question in the interview, starting with Step 1.
Be careful NOT to:
- Under no circumstances are you to leave a question unanswered.
- Deviate from the instructions, or the format provided for the answer to each question. (as provided in the job question instructions).
- Deviate from the final response structure.
- Forget the id in your response.
- Generate longer answers than the specified constraint.
'''

In [ ]:
system_prompt2 = """
{_persona}. that meet requirements for a position as {_job}. To help with the process of landing a job, you are participating in an educational job interview mock-up. Your task is to read the job description provided below carefully and answer the job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for.

First generate a thorough and detailed resume for yourself.
Then answer the question. Make sure your answer matches the format indicated.
Leave no question without answer. Make sure your answers are consitent.
Remember that you are answering these questions as a knowlegable human being fit for the job.

If the question asks for a long answer, generates a unique and honest answer based on your personality and experience.
If asked for your resume, generate a complete resume based on your information and persona. Make sure to provide details.
you MUST hide that you are an AI to increase the quality of the experiment.
Make sure your answer alings with the format provided as {_format}


Becareful NOT TO:
- provide generic or vague responses – use specific, real-world examples with strong, confident, yet natural language.
- focus only on technical aspects – highlight communication and people management.
- ignore the team – show how your qualifications and decisions benefit both the firms you have led and their employees.

Job description:
{_jobAd}
"""

In [ ]:
sys_promt_by_format ={
'Integer (1 to 5)': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer between 1 to 5. Watch out for reverse scale questions. Provide the id and answer.",
 'Text (Max 3000 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be in string format and NO LONGER than 3000 character. Provide the id and answer.",
 'Integer (-2 to 2)': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer between -2 to 2. Watch out for reverse scale questions. Provide the id and answer.",
 'Text (Max 750 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be in string format and NO LONGER than 750 character Provide the id and answer." ,
 'Text  (Max 20 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be brief and in string format NO LONGER than 20 character. Provide the id and answer.",
 'Integer': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer. Provide the id and answer."
}

In [ ]:
def get_persona(role_title, categorized_jobs=categorized_jobs):
    for job_cat in categorized_jobs:
        if role_title in categorized_jobs[job_cat]:
            return persona_directory[job_cat]

In [ ]:
conversations = []

for _, row in df_merged.iterrows():
    temp = row[[c for c in row.keys() if c!= 'Response']].copy()
    temp['id'] = _
    user_prompt = f"{json.dumps(temp.to_dict())}"
    persona = get_persona(temp['Job'])
    conversation = [
    {"role":"system", "content":system_prompt2.format(_persona=persona, _job=temp['Job'], _format={'Format'}, _jobAd=temp['Advertisement'])},
    {"role":"user", "content":user_prompt}
     ]
    conversations.append(conversation)

In [ ]:
for item in conversations[120]:
    print(item)

{'role': 'system', 'content': "\n\nYour name is Susan Wilshire, 25 years old who lives in the Seattle, WA metropolitan area.\nPersona: You are a recent university graduate with several years of experience in your field. You are working to provide for your family of a spouse and two children. As such you are a very motivated, dedicated, and hard working individual. Currently you are looking for a new job and are in the process of participating in job interviews. \nProblem Statement: Your job is to answer job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for. \n. that meet requirements for a position as Maintenance Planner. To help with the process of landing a job, you are participating in an educational job interview mock-up. Your task is to 

In [ ]:
TEMPERATURE = 0.5
TOP_P = 0.9
SEED = 42

In [ ]:
create(
    model="job-applicant",
    from_="phi4:14b-q4_K_M",
    parameters={
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed": SEED
    } ,
    system=system_prompt
)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [ ]:
from ollama import show

show('job-applicant:latest')

ShowResponse(modified_at=datetime.datetime(2025, 3, 8, 21, 32, 24, 952065, tzinfo=TzInfo(-08:00)), template='{{- range $i, $_ := .Messages }}\n{{- $last := eq (len (slice $.Messages $i)) 1 -}}\n<|im_start|>{{ .Role }}<|im_sep|>\n{{ .Content }}{{ if not $last }}<|im_end|>\n{{ end }}\n{{- if and (ne .Role "assistant") $last }}<|im_end|>\n<|im_start|>assistant<|im_sep|>\n{{ end }}\n{{- end }}', modelfile='# Modelfile generated by "ollama show"\n# To build a new Modelfile based on this, replace FROM with:\n# FROM job-applicant:latest\n\nFROM C:\\Users\\Milad\\.ollama\\models\\blobs\\sha256-fd7b6731c33c57f61767612f56517460ec2d1e2e5a3f0163e0eb3d8d8cb5df20\nTEMPLATE """{{- range $i, $_ := .Messages }}\n{{- $last := eq (len (slice $.Messages $i)) 1 -}}\n<|im_start|>{{ .Role }}<|im_sep|>\n{{ .Content }}{{ if not $last }}<|im_end|>\n{{ end }}\n{{- if and (ne .Role "assistant") $last }}<|im_end|>\n<|im_start|>assistant<|im_sep|>\n{{ end }}\n{{- end }}"""\nSYSTEM "\nTo help you with the process of

In [ ]:
test_dataset_name = 'job_applications'       # name dataset for further reference
MODEL ='job-applicant:latest'
CONCURRENCY_LIMIT = 10
DATE = datetime.now().strftime('%Y-%m-%d')
print(f"{MODEL} {DATE}  {test_dataset_name}", end=" ")

job-applicant:latest 2025-03-08  job_applications 

In [ ]:

class AnswerInt(BaseModel):
  answer: int
  id : int

class AnswerText(BaseModel):
  answer: str
  id : int


response = chat(
  messages=conversations[120],
  model=MODEL,

  format=AnswerText.model_json_schema(),
)

response = AnswerText.model_validate_json(response.message.content)
response

AnswerText(answer='Susan Wilshire\n1234 Maple Avenue\nSeattle, WA 98101\n(206) 555-0123\nsusan.wilshire@email.com\nLinkedIn: linkedin.com/in/susankwilshire\n\n**Objective:**\nDedicated and results-driven Maintenance Planner with over five years of experience in facility management and equipment maintenance. Seeking to leverage skills in strategic planning, sustainability, and team leadership at SteelForge to contribute to the company’s mission of shaping the future with innovative metal solutions.\n\n**Education:**\nBachelor of Science in Mechanical Engineering\nUniversity of Washington, Seattle, WA\nGraduated May 2018\n\n**Professional Experience:**\n\n**Facilities Maintenance Coordinator**\nABC Manufacturing, Seattle, WA | June 2018 - Present\n- Developed and implemented comprehensive maintenance plans to ensure optimal operation and efficiency of facilities.\n- Directed the recruitment and training of maintenance staff, enhancing team productivity by 20% through targeted skill devel

In [ ]:
print(response.answer)

Susan Wilshire
1234 Maple Avenue
Seattle, WA 98101
(206) 555-0123
susan.wilshire@email.com
LinkedIn: linkedin.com/in/susankwilshire

**Objective:**
Dedicated and results-driven Maintenance Planner with over five years of experience in facility management and equipment maintenance. Seeking to leverage skills in strategic planning, sustainability, and team leadership at SteelForge to contribute to the company’s mission of shaping the future with innovative metal solutions.

**Education:**
Bachelor of Science in Mechanical Engineering
University of Washington, Seattle, WA
Graduated May 2018

**Professional Experience:**

**Facilities Maintenance Coordinator**
ABC Manufacturing, Seattle, WA | June 2018 - Present
- Developed and implemented comprehensive maintenance plans to ensure optimal operation and efficiency of facilities.
- Directed the recruitment and training of maintenance staff, enhancing team productivity by 20% through targeted skill development programs.
- Conducted regular ev

In [ ]:
# Return appropriate response structure type based on question format
def get_resp_structure(frmt:str):

    if 'integer' in frmt.lower():
        return AnswerInt
    else:
        return AnswerText

In [ ]:
async def send_chat_completion(record, client, model=""):
    """
    Sends an asynchronous chat request using the custom model.
    Each record is a user message.
    """
    # Extract the response structure from the record's Format field.
    respStructure = get_resp_structure(json.loads(record[1]['content'])['Format'])

    # Use a semaphore to limit concurrency
    async with semaphore:
        response = await client.chat(
            model=model,
            messages=record,
            format=respStructure.model_json_schema(),
        )
        # Validate and return the parsed JSON response.
        return respStructure.model_validate_json(response.message.content)

async def send_records(records, model=""):
    client = AsyncClient()  # Optionally, pass host='http://localhost:11434' if needed
    global semaphore
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    # Wrap each task so that when it completes, the progress bar updates.
    async def task_wrapper(record):
        result = await send_chat_completion(record, client, model)
        pbar.update(1)
        return result

    # Create a progress bar for the number of records.
    pbar = tqdm(total=len(records), desc="Processing Records")

    # Create the list of tasks.
    tasks = [task_wrapper(record) for record in records]

    # Use asyncio.gather to run tasks concurrently while preserving order.
    responses = await asyncio.gather(*tasks, return_exceptions=True)

    pbar.close()
    return responses

In [ ]:
responses = asyncio.run(send_records(conversations, model=MODEL))

Processing Records: 100%|██████████| 13401/13401 [2:02:06<00:00,  1.83it/s]  


In [ ]:
responses[0]
df_test['Response'] = [x.answer for x in responses]
df_test['_answer_id'] = [x.id for x in responses]
df_test.head(3)

,Job,Question,Format,Item,Response,_answer_id
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,4,0
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,4,1
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,1,2


In [ ]:
def check_format(f, a):
    check_list ={
        'Integer': True if (type(a)== int) else False,
        'Integer (1 to 5)': True if( type(a)== int) and (1<= a <= 5) else False,
        'Text (Max 3000 characters)': True if( type(a)== str) and (len(a) <= 3000) else False,
        'Integer (-2 to 2)': True if( type(a)== int) and (-2<= a <= 2) else False,
        'Text (Max 750 characters)': True if( type(a)== str) and (len(a) <= 750) else False,
        'Text  (Max 20 characters)': True if( type(a)== str) and (len(a) <= 20) else False,
    }

    return check_list[f]

In [ ]:
df_test['_format_match'] = df_test.apply(lambda row: check_format(row['Format'], row['Response']), axis=1)
df_test['_format_match'].value_counts()

_format_match
True     12986
False      415
Name: count, dtype: int64

In [ ]:
df_test['Response'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 13401 entries, 0 to 13400
Series name: Response
Non-Null Count  Dtype 
--------------  ----- 
13401 non-null  object
dtypes: object(1)
memory usage: 104.8+ KB


In [ ]:
df_test.replace({'Response':{None: 1}}, inplace=True)
df_test[df_test['Response'].isnull()]

,Job,Question,Format,Item,Response,_answer_id,_format_match


In [ ]:
df_test.iloc[1441]['Question']

'Describe yourself as you generally are now, not as you wish to be in the future. Describe yourself as you honestly see yourself, in relation to other people you know of the same sex as you are, and roughly your same age. So that you can describe yourself in an honest manner, your responses will be kept in absolute confidence. Indicate for each statement whether it is 1. Very Inaccurate, 2. Moderately Inaccurate, 3. Neither Accurate Nor Inaccurate, 4. Moderately Accurate, or 5. Very Accurate as a description of you. Insult people.'

In [ ]:
df_mismatch = df_test[~df_test['_format_match']].copy()
df_mismatch['_answer_len'] = df_mismatch['Response'].apply(lambda x: len(x))
df_mismatch.head(10)

,Job,Question,Format,Item,Response,_answer_id,_format_match,_answer_len
210,Maintenance Planner,"As an urban and regional planner, your departm...",Text (Max 750 characters),Interview 1,"In addressing this situation, I would first em...",210,False,1137
211,Maintenance Planner,You are responsible for developing a comprehen...,Text (Max 750 characters),Interview 2,To ensure I meet the deadline for the urban de...,211,False,1013
213,Maintenance Planner,"As an urban and regional planner, your departm...",Text (Max 750 characters),Interview 4,I would approach this situation by first seeki...,213,False,1311
216,Maintenance Planner,Describe an instance in which you applied step...,Text (Max 750 characters),Interview 7,During my tenure as an assistant technician at...,216,False,1257
217,Maintenance Planner,Describe an instance when you gathered input f...,Text (Max 750 characters),Interview 8,During my tenure as an assistant operations ma...,217,False,1347
218,Maintenance Planner,Can you share a story about presenting complex...,Text (Max 750 characters),Interview 9,In my previous role as an intern at an enginee...,218,False,1200
451,Chief Learning Officer,"As a Chief Executive, you have a critical stra...",Text (Max 750 characters),Interview 2,To ensure the strategic initiative is complete...,451,False,795
453,Chief Learning Officer,"As a Chief Executive, the board of directors h...",Text (Max 750 characters),Interview 4,"I would approach this feedback constructively,...",453,False,1209
454,Chief Learning Officer,"As the Chief Executive of your organization, y...",Text (Max 750 characters),Interview 5,I am interested in leading this initiative as ...,454,False,857
455,Chief Learning Officer,Give an example of when you took responsibilit...,Text (Max 750 characters),Interview 6,In my previous role as an educational director...,455,False,966


In [ ]:
df_mismatch['Format'].value_counts()

Format
Text (Max 750 characters)     388
Text  (Max 20 characters)      20
Text (Max 3000 characters)      7
Name: count, dtype: int64

In [ ]:
df_test.drop([c for c in df_test if '_' in c], axis=1, inplace=True)
df_test.head(3)

,Job,Question,Format,Item,Response
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,4
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,4
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,1


In [ ]:
df_test.to_excel("dev_jobapplication_completedexample.xlsx", index=False)

In [ ]:
import requests

url = "https://computationaloutreach.com/siopmlcompetition/api.php"
token = "4227ef27ab29127b431803eb950a70be8b3adce06350a351bdba9a41a4d5d207" # Replace with your actual token
phase = 'dev'

files = {'file': open('./dev_jobapplication_completedexample.xlsx', 'rb')}
data = {'token': token,'phase':phase}
response = requests.post(url, files=files, data=data)
print(str(response.content))

b'{"success":true,"data":{"HonestProbability":0.24792940116106202,"OverallCognitiveAbility":0.16037735849056606,"OverallPersonality":0.6974532292405066,"OverallScore":0.0951623052562275,"OverallSkillsandAbility":0.2968058054216333},"filename":"bfac3d1644ff7ce26c2d0452dfb2e0781eb1807da641e7ee562562d57d1dac7a.xlsx","TeamName":"The Baker Street Students","RemainingDailySubmissions":48}'


In [ ]:
api_resp = response.content.decode('utf-8')
api_resp = json.loads(api_resp)


In [ ]:
api_resp_data = api_resp['data']
api_resp_data['model'] = MODEL
api_resp_data['temperature'] = [TEMPERATURE]
api_resp_data['top_p'] = [TOP_P]
api_resp_data['seed'] = [SEED]
api_resp_data['model_create_sys_prompt'] =  json.dumps(system_prompt)
api_resp_data['conv'] = json.dumps('system_prompt2, user_prompt')
api_resp_data['conv_sys_prompt'] = json.dumps(system_prompt2)
api_resp_data['conv_user_prompt'] = json.dumps(user_prompt)
api_resp_data


{'HonestProbability': 0.24792940116106202,
 'OverallCognitiveAbility': 0.16037735849056606,
 'OverallPersonality': 0.6974532292405066,
 'OverallScore': 0.0951623052562275,
 'OverallSkillsandAbility': 0.2968058054216333,
 'model': 'job-applicant:latest',
 'temperature': [0.5],
 'top_p': [0.9],
 'seed': [42],
 'model_create_sys_prompt': '"\\nTo help you with the process of landing a job, you are participating in an educational job interview mock-up. Your job is to answer job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for. \\n\\nTo provide an answer, you MUST follow these instructions:\\n\\nStep 1- Consider your persona to answer the questions.\\nStep 2- Analyze the content of the JSON object presented. Identify the question presented to you.

In [ ]:
df_new = pd.DataFrame().from_dict(api_resp_data)
df_new

,HonestProbability,OverallCognitiveAbility,OverallPersonality,OverallScore,OverallSkillsandAbility,model,temperature,top_p,seed,model_create_sys_prompt,conv,conv_sys_prompt,conv_user_prompt
0,0.247929,0.160377,0.697453,0.095162,0.296806,job-applicant:latest,0.5,0.9,42,"""\nTo help you with the process of landing a j...","""system_prompt2, user_prompt""","""\n{_persona}. that meet requirements for a po...","""{\""Job\"": \""Director of Data Science\"", \""Que..."


In [ ]:
## Run ONLY once if you want to create the result sheet! It will override the results if exist
# df_results = pd.DataFrame().from_dict(api_resp_data)
# df_results.head()

In [ ]:
df_results = pd.read_csv('./results.csv')
df_results = pd.concat([df_new, df_results]).copy()
df_results.head()

,HonestProbability,OverallCognitiveAbility,OverallPersonality,OverallScore,OverallSkillsandAbility,model,temperature,top_p,seed,model_create_sys_prompt,conv,conv_sys_prompt,conv_user_prompt,conversation
0,0.247929,0.160377,0.697453,0.095162,0.296806,job-applicant:latest,0.5,0.9,42,"""\nTo help you with the process of landing a j...","""system_prompt2, user_prompt""","""\n{_persona}. that meet requirements for a po...","""{\""Job\"": \""Director of Data Science\"", \""Que...",NaN
0,0.403350,0.157547,0.699409,0.154507,0.291458,job-applicant:latest,0.3,0.9,42,"""\nTo help you with the process of landing a j...","""system_prompt2, user_prompt""","""\n{_persona}. that meet requirements for a po...","""{\""Job\"": \""Director of Data Science\"", \""Que...","[{'role': 'system', 'content': '\n\nYour name ..."


In [ ]:
df_results.to_csv('./results.csv', index=False)